# 02 · Tratamento e Preparação dos Dados
Aplica as decisões levantadas em `01_eda`: remoção de duplicatas, engenharia de atributos e split treino/teste — **nesta ordem**, para evitar vazamento de dados (data leakage).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/creditcard.csv')
print('Antes:', df.shape)

Antes: (284807, 31)


## Remoção de duplicatas

In [2]:
dups = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f'{dups} linhas duplicadas removidas. Formato atual: {df.shape}')

1081 linhas duplicadas removidas. Formato atual: (283726, 31)


## Engenharia de atributos

- `Amount_log`: log1p do valor da transação (reduz o efeito de outliers extremos).
- `Hour`: hora do dia derivada de `Time` (mais informativa que um contador corrido de segundos).
- Colunas originais `Amount` e `Time` são **removidas** após a transformação, para não manter features redundantes/correlacionadas no mesmo modelo.

In [3]:
df['Amount_log'] = np.log1p(df['Amount'])
df['Hour'] = (df['Time'] % (24 * 3600)) // 3600
df = df.drop(columns=['Amount', 'Time'])
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V22,V23,V24,V25,V26,V27,V28,Class,Amount_log,Hour
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0,5.014760,0.0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0,1.305626,0.0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0,5.939276,0.0
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0,4.824306,0.0
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0,4.262539,0.0


## Split treino/teste — **antes** de qualquer normalização

Ponto crítico de correção em relação à primeira versão do notebook: o `StandardScaler` **não** deve ser ajustado (`fit`) no dataset inteiro. Ele precisa aprender média/desvio-padrão apenas do treino, e aplicar essa transformação (`transform`) no teste — senão o modelo "vê" estatísticas do conjunto de teste antes da avaliação (vazamento de dados), o que infla artificialmente as métricas.

In [4]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=42
)

print('Treino:', X_train.shape, ' | Proporção de fraude:', y_train.mean().round(5))
print('Teste :', X_test.shape, ' | Proporção de fraude:', y_test.mean().round(5))

Treino: (198608, 30)  | Proporção de fraude: 0.00167
Teste : (85118, 30)  | Proporção de fraude: 0.00167


**Nota sobre balanceamento (SMOTE / undersampling)**: a técnica de balanceamento é aplicada *dentro* do notebook de modelagem (`03_modelagem`), e sempre **somente sobre o conjunto de treino** — nunca sobre o teste, e nunca antes do split. Isso evita simular fraudes sintéticas que "vazam" para a avaliação final.

## Persistência dos conjuntos processados

In [5]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
print('Conjuntos salvos em data/processed/')

Conjuntos salvos em data/processed/


Segue para `03_modelagem.ipynb`.